In [2]:
import torch
import numpy as np
from PIL import Image
import pdb
import os
import cv2
from scipy.io import loadmat,savemat
from collections import defaultdict
import tqdm

In [3]:
data_path = "/home/zhengwei/Desktop/Zhengwei/Projects/datasets/SKSF-A"
train_style = ['1', '2', '3', '4', '5', '6', '7']
files_rgb = os.listdir(data_path+'/Photo')
files_sk = {s: os.listdir(f'{data_path}/style{s}') for s in train_style}

In [4]:
print(files_rgb)
print(files_sk.keys())

['95.png', '105.png', '64.png', '97.png', '131.png', '1.png', '93.png', '128.png', '15.png', '47.png', '69.png', '23.png', '134.png', '74.png', '133.png', '56.png', '76.png', '43.png', '132.png', '51.png', '22.png', '123.png', '119.png', '57.png', '117.png', '2.png', '87.png', '89.png', '53.png', '85.png', '109.png', '10.png', '61.png', '91.png', '92.png', '84.png', '113.png', '59.png', '27.png', '127.png', '40.png', '63.png', '68.png', '106.png', '94.png', '39.png', '126.png', '111.png', '31.png', '100.png', '122.png', '30.png', '6.png', '78.png', '80.png', '19.png', '35.png', '116.png', '55.png', '8.png', '26.png', 'Thumbs.db', '62.png', '45.png', '3.png', '86.png', '28.png', '65.png', '88.png', '12.png', '101.png', '60.png', '50.png', '52.png', '108.png', '99.png', '71.png', '107.png', '129.png', '21.png', '66.png', '124.png', '5.png', '20.png', '48.png', '58.png', '115.png', '38.png', '103.png', '29.png', '125.png', '110.png', '98.png', '4.png', '120.png', '11.png', '14.png', '25.p

In [5]:
pid_container = set()
for s in files_sk.keys():
    files = files_sk[s]
    for img_path in files:
        if img_path.endswith('.png') or img_path.endswith('.PNG'):
            pid = int(img_path[:-4])
        pid_container.add(pid)
pid2label = {pid:label for label, pid in enumerate(pid_container)}


_train_image = defaultdict(list)
_train_sketch = defaultdict(list)

sketch2idx = {s: i for i, s in enumerate(train_style)}

for s in files_sk.keys():
    train_image = files_sk[s]
    for img_path in train_image:
        if img_path.endswith('.png') or img_path.endswith('.PNG'):
            pid = int(img_path[:-4])
        _train_image[pid].append(f'style{s}/{img_path}')
        _train_sketch[pid].append(sketch2idx[s])


In [6]:
print(sorted(_train_sketch.keys()))
print(_train_sketch)
# count the min length of images list in each id
min_len = 100000
for key in _train_image.keys():
    min_len = min(min_len, len(_train_image[key]))
print(min_len)


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134]
defaultdict(<class 'list'>, {121: [0, 1, 2, 3, 4, 5, 6], 115: [0, 1, 2, 3, 4, 5, 6], 7: [0, 1, 2, 3, 4, 5, 6], 83: [0, 1, 2, 3, 4, 5, 6], 120: [0, 1, 2, 3, 4, 5, 6], 122: [0, 1, 2, 3, 4, 5, 6], 19: [0, 1, 2, 3, 4, 5, 6], 129: [0, 1, 2, 3, 4, 5, 6], 51: [0, 1, 2, 3, 4, 5, 6], 81: [0, 1, 2, 3, 4, 5, 6], 11: [0, 1, 2, 3, 4, 5, 6], 21: [0, 1, 2, 3, 4, 5, 6], 55: [0, 1, 2, 3, 4, 5, 6], 35: [0, 1, 2, 3, 4, 5, 6], 113: [0, 1, 2, 3, 4, 5, 6]

In [7]:
import shutil

def copy_file(src_path, dst_path):
    # Get the directory part of the destination path
    dst_dir = os.path.dirname(dst_path)
    
    # Create the directory if it doesn't exist
    if not os.path.exists(dst_dir):
        os.makedirs(dst_dir)
    
    shutil.copy(src_path, dst_path)

In [14]:
# seed manual seed for random function
seed = 0
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
np.random.seed(seed)

few_shot_basepath = '/home/zhengwei/Desktop/Zhengwei/Projects/datasets/SKSF-A/fewshot/'

for pid in tqdm.tqdm(sorted(_train_image.keys())):
    if not pid in pid2label.keys():
            continue
    img_paths = _train_image[pid]
    sketch_idx = _train_sketch[pid]

    # print(img_paths)

    # # select a random number from 0 to len(sketch_idx)
    # select = np.random.randint(0, len(sketch_idx))

    # select 2 random numbers from 0 to len(sketch_idx)
    select = np.random.choice(len(sketch_idx), 5, replace=False)
    # selected_sketch = sketch_idx[select]
    # unselected_sketch = [sketch_idx[i] for i in range(len(sketch_idx)) if i != select]

    # print(sketch_idx)
    # print(selected_sketch)
    # print(unselected_sketch)
    
    for idx, img_path in enumerate(img_paths):
        style = train_style[sketch_idx[idx]]
        if img_path.endswith('.png') or img_path.endswith('.PNG'):
            img = img_path.split('/')[-1][:-4]
            img = f'{int(img):04d}'  # Ensure img is a 4-digit string
        else:
             print(f'Error: {img_path} is not a .png file')
             continue
        if idx in select:
            # copy image_file from one path to another, with out open it
            copy_file(f'{data_path}/{img_path}', f'{few_shot_basepath}/5sketch/finetune/{img}_{style}.jpg')
        else:
            copy_file(f'{data_path}/{img_path}', f'{few_shot_basepath}/5sketch/test/{img}_{style}.jpg')




 78%|███████▊  | 105/134 [00:00<00:00, 521.92it/s]

Error: style2/Thumbs.db is not a .png file
Error: style6/Thumbs.db is not a .png file
Error: style1/Thumbs.db is not a .png file
Error: style3/Thumbs.db is not a .png file
Error: style4/Thumbs.db is not a .png file
Error: style5/Thumbs.db is not a .png file
Error: style7/Thumbs.db is not a .png file


100%|██████████| 134/134 [00:00<00:00, 518.21it/s]


In [15]:
import os

def print_directory_structure(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for idx, f in enumerate(sorted(files)):
            if idx < 6:
                print('{}{}'.format(subindent, f))

# Replace 'your_directory_path' with the path of the directory you want to print
print_directory_structure(few_shot_basepath)

/
1sketch/
    finetune/
        0001_7.jpg
        0002_2.jpg
        0003_7.jpg
        0004_6.jpg
        0005_3.jpg
        0006_5.jpg
    test/
        0001_1.jpg
        0001_2.jpg
        0001_3.jpg
        0001_4.jpg
        0001_5.jpg
        0001_6.jpg
5sketch/
    finetune/
        0001_1.jpg
        0001_2.jpg
        0001_3.jpg
        0001_4.jpg
        0001_7.jpg
        0002_1.jpg
    test/
        0001_5.jpg
        0001_6.jpg
        0002_3.jpg
        0002_6.jpg
        0003_1.jpg
        0003_5.jpg
2sketch/
    finetune/
        0001_3.jpg
        0001_7.jpg
        0002_1.jpg
        0002_2.jpg
        0003_4.jpg
        0003_7.jpg
    test/
        0001_1.jpg
        0001_2.jpg
        0001_4.jpg
        0001_5.jpg
        0001_6.jpg
        0002_3.jpg


In [16]:
import os
import pandas as pd
from tabulate import tabulate

def file_structure(path):
    file_dict = {}
    for root, dirs, files in os.walk(path):
        # Use the relative path as the header, with 'Base' indicating the root directory
        relative_path = os.path.relpath(root, path) if os.path.relpath(root, path) != '.' else 'Base'
        file_dict[relative_path] = len(files)
    return file_dict

file_dict = file_structure(few_shot_basepath)

# Creating a DataFrame from the dictionary, with directory structures as columns
df = pd.DataFrame([file_dict])

# Optional: If you want to sort the columns to reflect the directory structure hierarchy
# df = df.sort_index(axis=1)

# Printing the table using tabulate
print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))


+--------+-----------+--------------------+----------------+-----------+--------------------+----------------+-----------+--------------------+----------------+
|   Base |   1sketch |   1sketch/finetune |   1sketch/test |   5sketch |   5sketch/finetune |   5sketch/test |   2sketch |   2sketch/finetune |   2sketch/test |
|--------+-----------+--------------------+----------------+-----------+--------------------+----------------+-----------+--------------------+----------------|
|      0 |         0 |                133 |            805 |         0 |                668 |            270 |         0 |                267 |            671 |
+--------+-----------+--------------------+----------------+-----------+--------------------+----------------+-----------+--------------------+----------------+


In [42]:
# according to the files in /home/zhengwei/Desktop/Zhengwei/Projects/datasets/Market-Sketch-1K/sketch/fewshot/2sketch
# adjust copy file from  /home/zhengwei/Desktop/Zhengwei/Projects/datasets/Market-Sketch-1K/tensor/CLIPreidNew/sketch/fewshot/all to  /home/zhengwei/Desktop/Zhengwei/Projects/datasets/Market-Sketch-1K/tensor/CLIPreidNew/sketch/fewshot/2sketch

